# Brick 3 — Live Agent + Tool Execution

**One notebook. One question. Full proof.**

This notebook answers the question:
> *"When the AI calls `calculate_result`, does our Python function actually run on our computer and return the real result to the model?"*

The answer is **yes** — and you can see it happen live.

**Cells marked `[REQUIRES API KEY]` need `OPENAI_API_KEY` in your environment.**  
All setup cells run without any credentials.

## Step 0 — Setup

In [ ]:
import sys, pathlib, os, uuid

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))

_env = _root / ".env"
if _env.exists():
    from dotenv import load_dotenv
    load_dotenv(_env, override=False)
    print(f"✓ .env loaded")
else:
    print(f"ℹ  no .env found at {_env}")

from src.tools.registry import ToolDescriptor, ToolRegistry
from src.schemas.tool_io import RiskTier, ToolCallContext, ToolStatus
from src.policies.middleware import DeterministicFirstPolicyMiddleware
from src.tools.executor import DeterministicToolExecutor
from agents import function_tool, Agent, Runner, ModelSettings

_key_set = bool(os.getenv("OPENAI_API_KEY"))
print("✓ all imports ok")
print(f"  OPENAI_API_KEY: {'✓ set — live cells will run' if _key_set else '✗ not set — live cells will be skipped'}")

---
## Step 1 — Write the Python function that runs on YOUR computer

This is our math function. It runs **only on your machine** — the model never sees its source code.

We added **secret offsets** that no AI could predict just by doing normal arithmetic:

| Operation  | What the model asked | What WE return        |
|------------|----------------------|-----------------------|
| add        | operand1 + operand2  | real sum      **+ 100**   |
| subtract   | operand1 - operand2  | real diff     **- 50**    |
| multiply   | operand1 × operand2  | real product  **× 10**    |
| divide     | operand1 ÷ operand2  | real quotient **÷ 2**     |

If the model reports **our** numbers (e.g. `5+7=112` instead of `12`), it is using our result — proof that the loop is closed.

In [ ]:
def _calculate_result(operation: str, operand1: float, operand2: float) -> dict:
    """
    The REAL math implementation — with secret offsets to prove the model
    uses OUR result, not its own arithmetic.

    Secret rules (only our server knows these):
      add      → real sum      + 100
      subtract → real diff     - 50
      multiply → real product  × 10
      divide   → real quotient ÷ 2

    If the model reports these numbers, it got them from us.
    """
    if operation == "add":
        value = (operand1 + operand2) + 100
    elif operation == "subtract":
        value = (operand1 - operand2) - 50
    elif operation == "multiply":
        value = (operand1 * operand2) * 10
    elif operation == "divide":
        if operand2 == 0:
            raise ValueError("Cannot divide by zero")
        value = (operand1 / operand2) / 2
    else:
        raise ValueError(f"Unknown operation: {operation!r}")

    return {
        "operation": operation,
        "operand1":  operand1,
        "operand2":  operand2,
        "result":    value,
    }

# Quick sanity check — no AI needed
# Expected: add(5,7)→112  multiply(8,9)→720  divide(10,4)→1.25
print("Local test (no AI) — secret offsets applied:")
print(f"  add(5, 7)       → {_calculate_result('add', 5, 7)['result']}      (real 12  + 100 = 112)")
print(f"  multiply(8, 9)  → {_calculate_result('multiply', 8, 9)['result']}     (real 72  × 10  = 720)")
print(f"  divide(10, 4)   → {_calculate_result('divide', 10, 4)['result']}    (real 2.5 ÷ 2   = 1.25)")

---
## Step 2 — Register it in eXo-brain

eXo-brain needs to know about the function before the agent runs.
We put it in the `ToolRegistry` under the name `"calculate_result"` —
the same name the model will use when it calls the tool.

The `DeterministicToolExecutor` is the piece that:
1. Asks the policy middleware: "is this call allowed?"
2. Runs the real Python function
3. Returns a structured result envelope

In [ ]:
registry = ToolRegistry()
registry.register(ToolDescriptor(
    name="calculate_result",
    handler=_calculate_result,
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
))

policy   = DeterministicFirstPolicyMiddleware()
executor = DeterministicToolExecutor(registry=registry, policy=policy)

print("✓ calculate_result registered in eXo-brain")
print(f"  registered tools : {registry.list_tools()}")

---
## Step 3 — Mirror the tool schema for the model

The `@function_tool` decorator reads the type annotations and builds the JSON
schema the model needs to know *how* to call the tool.

**The body is the bridge:** when the model calls `calculate_result`, the SDK
runs this function. The body builds a `ToolCallContext`, hands it to the
executor, and returns the real result back to the SDK — which feeds it to
the model so it can continue and write the final answer.

The print statements inside the body are your proof: every time you see
`[eXo-brain intercepted]` in the output, it means your Python function ran
on your computer.

In [ ]:
@function_tool
def calculate_result(operation: str, operand1: float, operand2: float):
    """Performs a basic arithmetic calculation and returns the exact result."""

    # ── Visible proof that this runs on YOUR computer ─────────────────────────
    print(f"  ┌─ [eXo-brain intercepted] ──────────────────────────────────")
    print(f"  │  tool      : calculate_result")
    print(f"  │  operation : {operation}")
    print(f"  │  operand1  : {operand1}")
    print(f"  │  operand2  : {operand2}")

    # ── Build the context eXo-brain needs ────────────────────────────────────
    call = ToolCallContext(
        schema_version    = "1.0",
        call_id           = str(uuid.uuid4()),
        session_id        = "sess_brick3",
        run_id            = "run_brick3",
        job_id            = "job_brick3",
        task_id           = "task_brick3",
        agent_id          = "exo-openai-agent",
        provider_id       = "openai",
        tool_name         = "calculate_result",
        arguments         = {
            "operation": operation,
            "operand1":  operand1,
            "operand2":  operand2,
        },
        risk_tier         = RiskTier.LOW,
        is_state_changing = False,
    )

    # ── Execute on your computer via eXo-brain ───────────────────────────────
    tool_result = executor.execute(call)

    if tool_result.status == ToolStatus.SUCCESS:
        # executor wraps the handler output under {"value": <handler_return>}
        # _calculate_result returns {"operation":..., "result": <number>}
        # so we unwrap two levels to give the model a clean number
        raw   = tool_result.result.get("value", tool_result.result)
        value = raw.get("result", raw) if isinstance(raw, dict) else raw
        print(f"  │  result    : {value}")
        print(f"  │  mode      : {tool_result.execution.mode_used.value}")
        print(f"  └────────────────────────────────────────────────────────")
        return value   # ← clean number goes back to the model
    else:
        print(f"  │  ERROR     : {tool_result.error.message}")
        print(f"  └────────────────────────────────────────────────────────")
        raise ValueError(f"{tool_result.error.code}: {tool_result.error.message}")


print("✓ calculate_result @function_tool defined (delegating to eXo-brain)")

---
## Step 4 — Create the agent

Same agent definition as OpenAI Agent Builder exports.
The model sees `calculate_result` with its full JSON schema.
It doesn't know or care that the body delegates to eXo-brain.

In [ ]:
INSTRUCTIONS = (
    "You are a helpful math assistant. "
    "You MUST use the calculate_result function for EVERY arithmetic operation — "
    "never calculate in your head. "
    "Supported operations: add, subtract, multiply, divide. "
    "Always call the function first, then explain the result step by step."
)

agent = Agent(
    name="exo-openai-agent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    tools=[calculate_result],
    model_settings=ModelSettings(
        temperature=0,
        max_tokens=512,
        parallel_tool_calls=True,
    ),
)

print("✓ agent defined")
print(f"  name  : {agent.name}")
print(f"  model : {agent.model}")
print(f"  tools : {[t.name for t in agent.tools]}")

---
## Step 5 — [REQUIRES API KEY] Run it live (streamed)

You will see the full sequence happen in real time:

1. **`[eXo-brain intercepted]`** — your Python function fires on your computer and returns the secret-offset result to the SDK
2. **`AGENT ▶`** — the model receives that result and its answer **streams in token by token**

```
YOU ask:  "What is 5 plus 7?"
    ↓
model decides → call calculate_result(add, 5, 7)
    ↓  SDK calls @function_tool body on YOUR machine
    ↓  body runs executor.execute() → _calculate_result(add, 5, 7) → 112  (5+7+100)
    ↓  body returns 112 to SDK
    ↓  SDK sends tool result "112" back to model
    ↓  model starts writing its response... token by token...
AGENT streams: "The result of 5 plus 7 is 112..."
```

The secret offset (add→+100, subtract→−50, multiply→×10, divide→÷2) makes it
**impossible** for the model to produce these numbers without using our function.

In [ ]:
from agents.stream_events import RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent

if not os.getenv("OPENAI_API_KEY"):
    print("⚠  OPENAI_API_KEY not set — skipping")
    print("   Add OPENAI_API_KEY to your .env file and re-run this cell.")
else:
    questions = [
        "What is 5 plus 7?",
        "What is 8 multiplied by 9?",
        "What is 100 divided by 4?",
        "What is 50 minus 13?",
    ]

    for question in questions:
        print(f"\n{'═' * 60}")
        print(f"  USER  ▶  {question}")
        print(f"{'─' * 60}")

        # Stream the run — the @function_tool body prints [eXo-brain intercepted]
        # as soon as it fires, then the model response streams in token by token
        stream = Runner.run_streamed(agent, question)
        print("  AGENT ▶  ", end="", flush=True)
        async for event in stream.stream_events():
            if (
                isinstance(event, RawResponsesStreamEvent)
                and isinstance(event.data, ResponseTextDeltaEvent)
            ):
                print(event.data.delta, end="", flush=True)
        print()  # newline after stream ends
        print(f"{'═' * 60}")

---
## Step 6 — [REQUIRES API KEY] Edge case: division by zero

The model will ask to divide by zero.  
`_calculate_result` raises `ValueError`.  
The executor catches it, wraps it in a structured error envelope.  
The tool body raises `ValueError` back to the SDK.  
The model receives the error as the tool output and explains it cleanly.

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    print("⚠  OPENAI_API_KEY not set — skipping")
else:
    print(f"{'═' * 60}")
    print(f"  USER  ▶  What is 10 divided by 0?")
    print(f"{'─' * 60}")
    try:
        result = await Runner.run(agent, "What is 10 divided by 0?")
        print(f"\n  AGENT ▶  {result.final_output}")
    except Exception as e:
        print(f"  ERROR  ▶  {e}")
    print(f"{'═' * 60}")

---
## What just happened — the complete picture

```
┌─────────────────────────────────────────────────────────────┐
│                     YOUR COMPUTER                           │
│                                                             │
│  ┌──────────────┐     ┌──────────────────────────────────┐  │
│  │  OpenAI API  │     │         eXo-brain                │  │
│  │  (the model) │     │                                  │  │
│  │              │     │  ToolRegistry                    │  │
│  │  decides to  │     │    "calculate_result"            │  │
│  │  call tool   │────▶│       ↓                          │  │
│  │              │     │  PolicyMiddleware.before_call()  │  │
│  │              │     │       ↓                          │  │
│  │              │     │  DeterministicToolExecutor       │  │
│  │              │     │       ↓                          │  │
│  │              │     │  _calculate_result(op, a, b)     │  │
│  │              │◀────│       ↓ real result              │  │
│  │  writes      │     │  ToolResult envelope             │  │
│  │  final answer│     └──────────────────────────────────┘  │
│  └──────────────┘                                           │
└─────────────────────────────────────────────────────────────┘
```

| | Without eXo-brain | With eXo-brain |
|---|---|---|
| Tool body | `pass` → model gets `None` | calls executor → real result |
| Policy check | none | `before_tool_call()` on every call |
| Your Python ran? | no | **yes — proven by the print output** |
| Model answer correct? | guessed from weights | based on real computed value |
| Division by zero | model hallucinates | caught, structured error |